In [1]:
import pandas as pd
import numpy as np

In [3]:
df_qual = pd.read_parquet(r"C:\Users\Asus\Desktop\Formula1\data\bronze\qualifying\all_seasons_qualifying.parquet")
df_qual.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,quali_position,q1_time,q2_time,q3_time
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,HAM,44,mercedes,1,1:22.824,1:22.051,1:21.164
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,RAI,7,ferrari,2,1:23.096,1:22.507,1:21.828
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,VET,5,ferrari,3,1:23.348,1:21.944,1:21.838
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,max_verstappen,VER,33,red_bull,4,1:23.483,1:22.416,1:21.879
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,ricciardo,RIC,3,red_bull,5,1:23.494,1:22.897,1:22.152


- no **PRIMARY KEY**
- combination of season, round_number and driver_code is unique - **PRIMARY KEY**

In [4]:
df_qual.shape

(3455, 13)

In [5]:
df_qual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3455 entries, 0 to 3454
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   season           3455 non-null   int64 
 1   round_number     3455 non-null   int64 
 2   race_name        3455 non-null   object
 3   circuit_ref      3455 non-null   object
 4   race_date        3455 non-null   object
 5   driver_ref       3455 non-null   object
 6   driver_code      3455 non-null   object
 7   driver_number    3455 non-null   int64 
 8   constructor_ref  3455 non-null   object
 9   quali_position   3455 non-null   int64 
 10  q1_time          3455 non-null   object
 11  q2_time          2563 non-null   object
 12  q3_time          1702 non-null   object
dtypes: int64(4), object(9)
memory usage: 351.0+ KB


In [6]:
df_qual["qual_ref"] = df_qual["season"].astype(str).str[2:4] + "_" + df_qual["round_number"].astype(str) + "_" + df_qual["driver_ref"].astype(str)

In [7]:
df_qual.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,quali_position,q1_time,q2_time,q3_time,qual_ref
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,HAM,44,mercedes,1,1:22.824,1:22.051,1:21.164,18_1_hamilton
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,RAI,7,ferrari,2,1:23.096,1:22.507,1:21.828,18_1_raikkonen
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,VET,5,ferrari,3,1:23.348,1:21.944,1:21.838,18_1_vettel
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,max_verstappen,VER,33,red_bull,4,1:23.483,1:22.416,1:21.879,18_1_max_verstappen
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,ricciardo,RIC,3,red_bull,5,1:23.494,1:22.897,1:22.152,18_1_ricciardo


- changing datatype of race_date

In [8]:
df_qual["race_date"] = pd.to_datetime(df_qual["race_date"])

- handling q1_time, q2_time and q3_time columns

In [9]:
# problematic values in q1_time, q2_time, q3_time

a = df_qual[
    ~df_qual["q1_time"].astype(str).str.contains(":", na=False)
]["q1_time"].unique()

b = df_qual[
    ~df_qual["q2_time"].astype(str).str.contains(":", na=False)
]["q2_time"].unique()

c = df_qual[
    ~df_qual["q3_time"].astype(str).str.contains(":", na=False)
]["q3_time"].unique()

print("-------------------- q1_time --------------------")
print(a, "\n")

print("-------------------- q2_time --------------------")
print(b, "\n")

print("-------------------- q3_time --------------------")
print(c)

-------------------- q1_time --------------------
['' '53.904' '54.160' '54.037' '54.249' '54.236' '54.346' '54.388'
 '54.450' '54.207' '54.595' '54.309' '54.620' '54.301' '54.523' '54.194'
 '54.705' '54.796' '54.892' '54.963' '55.426'] 

-------------------- q2_time --------------------
[None '53.803' '53.819' '53.647' '53.825' '53.787' '53.856' '53.871'
 '53.818' '53.941' '53.840' '53.995' '54.026' '54.175' '54.377' '54.693'
 ''] 

-------------------- q3_time --------------------
[None '53.377' '53.403' '53.433' '53.613' '53.790' '53.906' '53.957'
 '54.010' '54.154' '54.200' '']


In [10]:
def qual_time_to_seconds(time_str):

    # Missing values
    if pd.isna(time_str) or str(time_str).strip() == "":
        return np.nan

    time_str = str(time_str)

    # Format: 1:22.824
    if ":" in time_str:
        mins, secs = time_str.split(":")
        return int(mins) * 60 + float(secs)

    # Format: 53.904
    return float(time_str)

In [11]:
time_cols = ["q1_time", "q2_time", "q3_time"]

for col in time_cols:
    df_qual[col + "_sec"] = df_qual[col].apply(qual_time_to_seconds)

- The number of values in **"q1_time"** and **"q1_time_sec"** differ because q1_time contains empty strings (''), which Pandas counts as non-null values, whereas the conversion function converts those empty strings to NaN, reducing the non-null count in q1_time_sec.

In [12]:
df_qual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3455 entries, 0 to 3454
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   season           3455 non-null   int64         
 1   round_number     3455 non-null   int64         
 2   race_name        3455 non-null   object        
 3   circuit_ref      3455 non-null   object        
 4   race_date        3455 non-null   datetime64[ns]
 5   driver_ref       3455 non-null   object        
 6   driver_code      3455 non-null   object        
 7   driver_number    3455 non-null   int64         
 8   constructor_ref  3455 non-null   object        
 9   quali_position   3455 non-null   int64         
 10  q1_time          3455 non-null   object        
 11  q2_time          2563 non-null   object        
 12  q3_time          1702 non-null   object        
 13  qual_ref         3455 non-null   object        
 14  q1_time_sec      3412 non-null   float64

- suspicious rows where Q2 or Q3 qualifying times are present, but the corresponding Q1 qualifying time is missing

In [13]:
df_qual[
    df_qual["q1_time_sec"].isna() &
    (
        df_qual["q2_time_sec"].notna() |
        df_qual["q3_time_sec"].notna()
    )
]

,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,quali_position,q1_time,q2_time,q3_time,qual_ref,q1_time_sec,q2_time_sec,q3_time_sec
2786,2024,15,Dutch Grand Prix,zandvoort,2024-08-25,albon,ALB,23,williams,10,,1:10.768,1:10.653,24_15_albon,NaN,70.768,70.653
2830,2024,17,Azerbaijan Grand Prix,baku,2024-09-15,gasly,GAS,10,alpine,15,,1:43.179,None,24_17_gasly,NaN,103.179,NaN


- dropping rows with inconsistent qualifying data where Q2/Q3 times exist but the Q1 time is missing

In [14]:
df_qual = df_qual[
    ~(
        df_qual["q1_time_sec"].isna() &
        (
            df_qual["q2_time_sec"].notna() |
            df_qual["q3_time_sec"].notna()
        )
    )
]

- q1_time, q2_time and q3_time are redundant columns now - dropping them

In [15]:
df_qual.drop(columns=["q1_time", "q2_time", "q3_time"], inplace=True)

- deriving new columns

In [16]:
# best qualifying time of the driver in a particular race

df_qual["best_quali_time_sec"] = df_qual["q3_time_sec"].fillna(df_qual["q2_time_sec"]).fillna(df_qual["q1_time_sec"])

In [17]:
# gap to pole

pole_times = (
    df_qual
    .groupby(["season", "round_number"])["best_quali_time_sec"]
    .min()
    .reset_index(name="pole_time_sec")
)


df_qual = df_qual.merge(
    pole_times,
    on=["season", "round_number"],
    how="left"
)

df_qual["gap_to_pole"] = (
    (df_qual["best_quali_time_sec"] - df_qual["pole_time_sec"])
)

df_qual.drop(columns=["pole_time_sec"], inplace=True)

df_qual.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,quali_position,qual_ref,q1_time_sec,q2_time_sec,q3_time_sec,best_quali_time_sec,gap_to_pole
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,HAM,44,mercedes,1,18_1_hamilton,82.824,82.051,81.164,81.164,0.000
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,RAI,7,ferrari,2,18_1_raikkonen,83.096,82.507,81.828,81.828,0.664
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,VET,5,ferrari,3,18_1_vettel,83.348,81.944,81.838,81.838,0.674
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,max_verstappen,VER,33,red_bull,4,18_1_max_verstappen,83.483,82.416,81.879,81.879,0.715
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,ricciardo,RIC,3,red_bull,5,18_1_ricciardo,83.494,82.897,82.152,82.152,0.988


In [18]:
# session progression flags 
df_qual["made_q2"] = df_qual["q2_time_sec"].notna()
df_qual["made_q3"] = df_qual["q3_time_sec"].notna()

In [19]:
df_qual.columns

Index(['season', 'round_number', 'race_name', 'circuit_ref', 'race_date',
       'driver_ref', 'driver_code', 'driver_number', 'constructor_ref',
       'quali_position', 'qual_ref', 'q1_time_sec', 'q2_time_sec',
       'q3_time_sec', 'best_quali_time_sec', 'gap_to_pole', 'made_q2',
       'made_q3'],
      dtype='object')

In [20]:
df_qual = df_qual[[ 'qual_ref', 'season', 'round_number', 'race_name', 'circuit_ref', 'race_date',
       'driver_ref', 'driver_code', 'driver_number', 'constructor_ref', 'quali_position',
       'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'best_quali_time_sec',
       'gap_to_pole', 'made_q2', 'made_q3']]

In [21]:
df_qual.to_parquet(r"C:\Users\Asus\Desktop\Formula1\data\silver\qualifying\cleaned_qualifying.parquet")

In [22]:
df_qual.head()

,qual_ref,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,quali_position,q1_time_sec,q2_time_sec,q3_time_sec,best_quali_time_sec,gap_to_pole,made_q2,made_q3
0,18_1_hamilton,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,HAM,44,mercedes,1,82.824,82.051,81.164,81.164,0.000,True,True
1,18_1_raikkonen,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,RAI,7,ferrari,2,83.096,82.507,81.828,81.828,0.664,True,True
2,18_1_vettel,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,VET,5,ferrari,3,83.348,81.944,81.838,81.838,0.674,True,True
3,18_1_max_verstappen,2018,1,Australian Grand Prix,albert_park,2018-03-25,max_verstappen,VER,33,red_bull,4,83.483,82.416,81.879,81.879,0.715,True,True
4,18_1_ricciardo,2018,1,Australian Grand Prix,albert_park,2018-03-25,ricciardo,RIC,3,red_bull,5,83.494,82.897,82.152,82.152,0.988,True,True


### Look at the pattern — every one of these rows has quali_position 19 or 20, and q1_time_sec through gap_to_pole are all NaN, with made_q2/made_q3 both False. This is almost certainly drivers who were knocked out in Q1 with no recorded lap time — typically because they:

- Had a mechanical failure / crash before setting a time (no lap = no time to record)
- Got an grid penalty placement that still shows them at P19/P20 in classification but with a blank time
- The session classification lists them as "last" by default ordering when no time was set

In [26]:
df_qual[df_qual["q1_time_sec"].isna()]

,qual_ref,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,quali_position,q1_time_sec,q2_time_sec,q3_time_sec,best_quali_time_sec,gap_to_pole,made_q2,made_q3
79,18_4_grosjean,2018,4,Azerbaijan Grand Prix,baku,2018-04-29,grosjean,GRO,8,haas,20,NaN,NaN,NaN,NaN,NaN,False,False
99,18_5_brendon_hartley,2018,5,Spanish Grand Prix,catalunya,2018-05-13,brendon_hartley,HAR,28,toro_rosso,20,NaN,NaN,NaN,NaN,NaN,False,False
119,18_6_max_verstappen,2018,6,Monaco Grand Prix,monaco,2018-05-27,max_verstappen,VER,33,red_bull,20,NaN,NaN,NaN,NaN,NaN,False,False
139,18_7_grosjean,2018,7,Canadian Grand Prix,villeneuve,2018-06-10,grosjean,GRO,8,haas,20,NaN,NaN,NaN,NaN,NaN,False,False
198,18_10_stroll,2018,10,British Grand Prix,silverstone,2018-07-08,stroll,STR,18,williams,19,NaN,NaN,NaN,NaN,NaN,False,False
199,18_10_brendon_hartley,2018,10,British Grand Prix,silverstone,2018-07-08,brendon_hartley,HAR,28,toro_rosso,20,NaN,NaN,NaN,NaN,NaN,False,False
496,19_4_raikkonen,2019,4,Azerbaijan Grand Prix,baku,2019-04-28,raikkonen,RAI,7,alfa,19,NaN,NaN,NaN,NaN,NaN,False,False
497,19_4_gasly,2019,4,Azerbaijan Grand Prix,baku,2019-04-28,gasly,GAS,10,red_bull,20,NaN,NaN,NaN,NaN,NaN,False,False
637,19_11_vettel,2019,11,German Grand Prix,hockenheimring,2019-07-28,vettel,VET,5,ferrari,20,NaN,NaN,NaN,NaN,NaN,False,False
677,19_13_kubica,2019,13,Belgian Grand Prix,spa,2019-09-01,kubica,KUB,88,williams,20,NaN,NaN,NaN,NaN,NaN,False,False


In [23]:
df_qual[df_qual["driver_code"] == "HAM"]

,qual_ref,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,quali_position,q1_time_sec,q2_time_sec,q3_time_sec,best_quali_time_sec,gap_to_pole,made_q2,made_q3
0,18_1_hamilton,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,HAM,44,mercedes,1,82.824,82.051,81.164,81.164,0.000,True,True
23,18_2_hamilton,2018,2,Bahrain Grand Prix,bahrain,2018-04-08,hamilton,HAM,44,mercedes,4,89.396,88.458,88.220,88.220,0.262,True,True
43,18_3_hamilton,2018,3,Chinese Grand Prix,shanghai,2018-04-15,hamilton,HAM,44,mercedes,4,93.283,91.914,91.675,91.675,0.580,True,True
61,18_4_hamilton,2018,4,Azerbaijan Grand Prix,baku,2018-04-29,hamilton,HAM,44,mercedes,2,102.693,102.676,101.677,101.677,0.179,True,True
80,18_5_hamilton,2018,5,Spanish Grand Prix,catalunya,2018-05-13,hamilton,HAM,44,mercedes,1,77.633,77.166,76.173,76.173,0.000,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3356,25_20_hamilton,2025,20,Mexico City Grand Prix,rodriguez,2025-10-26,hamilton,HAM,44,ferrari,3,76.736,76.458,75.938,75.938,0.352,True,True
3386,25_21_hamilton,2025,21,São Paulo Grand Prix,interlagos,2025-11-09,hamilton,HAM,44,ferrari,13,70.016,70.100,NaN,70.100,0.589,True,False
3412,25_22_hamilton,2025,22,Las Vegas Grand Prix,vegas,2025-11-23,hamilton,HAM,44,ferrari,20,117.115,NaN,NaN,117.115,9.181,False,False
3430,25_23_hamilton,2025,23,Qatar Grand Prix,losail,2025-11-30,hamilton,HAM,44,ferrari,18,80.907,NaN,NaN,80.907,1.520,False,False
